In [9]:
import sqlite3
import pandas as pd
import os

# ==== CONFIGURATION ====
db_path = "funnel.db"   # Path to your database file
table_name = "transactions" # Table you want to export
output_sql_file = "transactions.sql"

try:
    # 1. Connect to the database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 2. Read table into a Pandas DataFrame
    query = f"SELECT * FROM {table_name}"
    df = pd.read_sql_query(query, conn)

    # 3. Get CREATE TABLE statement
    cursor.execute(f"SELECT sql FROM sqlite_master WHERE type='table' AND name='{table_name}'")
    create_table_sql = cursor.fetchone()
    if not create_table_sql:
        raise ValueError(f"Table '{table_name}' not found in database.")
    create_table_sql = create_table_sql[0] + ";\n"

    # 4. Generate INSERT statements
    insert_statements = []
    for _, row in df.iterrows():
        values = []
        for val in row:
            if pd.isna(val):
                values.append("NULL")
            elif isinstance(val, str):
                values.append(f"'{val.replace("'", "''")}'")  # Escape single quotes
            else:
                values.append(str(val))
        insert_statements.append(f"INSERT INTO {table_name} VALUES ({', '.join(values)});")

    # 5. Write to .sql file
    with open(output_sql_file, "w", encoding="utf-8") as f:
        f.write(create_table_sql)
        f.write("\n".join(insert_statements))

    print(f"✅ SQL export completed: {os.path.abspath(output_sql_file)}")

except sqlite3.Error as e:
    print(f"Database error: {e}")
except Exception as e:
    print(f"Error: {e}")
finally:
    if 'conn' in locals():
        conn.close()


✅ SQL export completed: C:\Users\Jayaprakash\Downloads\customer-funnel-analysis-main\customer-funnel-analysis-main\notebooks\transactions.sql
